In [1]:
#%% stream rgb and depth (m)
import pyrealsense2 as rs
import numpy as np
import cv2
import matplotlib.pyplot as plt
from matplotlib.backends.backend_agg import FigureCanvasAgg as FigureCanvas
from matplotlib.figure import Figure
import io

# Setup
pipeline = rs.pipeline()
config = rs.config()
config.enable_stream(rs.stream.depth, 640, 480, rs.format.z16, 30)
config.enable_stream(rs.stream.color, 640, 480, rs.format.bgr8, 30)

# Start
pipeline.start(config)

# Align depth to color
align = rs.align(rs.stream.color)

# depth scale
depth_scale = pipeline.get_active_profile().get_device().first_depth_sensor().get_depth_scale()

print("Streaming aligned RGB and Depth. Press 'q' to quit.")

def create_depth_with_colorbar(depth, depth_scale, figsize=(6, 5)):
    """Create depth image with colormap and colorbar"""
    # Create figure
    fig = Figure(figsize=figsize, dpi=100)
    ax = fig.add_subplot(111)
    
    # Convert depth to meters for display
    depth_m = depth * depth_scale
    
    # Display with jet colormap
    im = ax.imshow(depth_m, cmap='jet', vmin=0, vmax=10)  # vmax=10 meters 
    ax.axis('off')
    
    # Add colorbar
    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label('Depth (m)', fontsize=10)
    
    # Render to numpy array
    canvas = FigureCanvas(fig)
    canvas.draw()
    
    # Convert to numpy array
    buf = canvas.buffer_rgba()
    img = np.asarray(buf)
    
    plt.close(fig)
    return img

try:
    while True:
        # Get frames and align
        frames = pipeline.wait_for_frames()
        aligned_frames = align.process(frames)
        
        # Get aligned frames
        color_frame = aligned_frames.get_color_frame()
        depth_frame = aligned_frames.get_depth_frame()
        
        if not color_frame or not depth_frame:
            continue
        
        # Convert to numpy
        color = np.asanyarray(color_frame.get_data())
        depth = np.asanyarray(depth_frame.get_data())
        
        # Create depth with colorbar
        depth_with_cbar = create_depth_with_colorbar(depth, depth_scale)
        
        # Display
        cv2.imshow("RGB", color)
        cv2.imshow("Depth", depth_with_cbar)
        
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

finally:
    pipeline.stop()
    cv2.destroyAllWindows()

Streaming aligned RGB and Depth. Press 'q' to quit.
